# Model Comparison

## Overview
We compare the top 5 models evaluated during the tuning phase: LightGBM, XGBoost, CatBoost, and two ensemble methods (Stacking Ridge and Weighted Average).

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import re

## 1. Load Leaderboard
We load the actual `final_leaderboard.csv` saved during model evaluation.

In [2]:
lb = pd.read_csv('../reports/final_leaderboard.csv')
display(lb)

,R2,RMSE,MAE,Model,Time
0,0.6016,10.0536,1.5216,LightGBM,95.4
1,0.6014,10.0566,1.5497,XGBoost,65.4
2,0.5970,10.1112,1.5566,CatBoost,242.7
3,0.5682,10.4661,1.8722,Stacking(Ridge),756.4
4,0.5614,10.5487,1.9214,SimpleAvg(3),0.0
5,0.5614,10.5485,1.9213,WeightedAvg(3),0.0


## 2. Update with Final CatBoost Metrics
The CatBoost model was selected and trained on the full final configuration. We extract its final metrics from the report.

In [3]:
with open('../reports/catboost_report.md', 'r') as f:
    text = f.read()

r2 = float(re.search(r'R2\*\*: ([\d.]+)', text).group(1))
rmse = float(re.search(r'RMSE\*\*: ([\d.]+)', text).group(1))
mae = float(re.search(r'MAE\*\*: ([\d.]+)', text).group(1))

# Update CatBoost row
cat_idx = lb[lb['Model'] == 'CatBoost'].index
lb.loc[cat_idx, 'R2'] = r2
lb.loc[cat_idx, 'RMSE'] = rmse
lb.loc[cat_idx, 'MAE'] = mae

lb = lb.sort_values(by='R2', ascending=False)
display(lb)

,R2,RMSE,MAE,Model,Time
2,0.604707,10.014333,1.421083,CatBoost,242.7
0,0.601600,10.053600,1.521600,LightGBM,95.4
1,0.601400,10.056600,1.549700,XGBoost,65.4
3,0.568200,10.466100,1.872200,Stacking(Ridge),756.4
4,0.561400,10.548700,1.921400,SimpleAvg(3),0.0
5,0.561400,10.548500,1.921300,WeightedAvg(3),0.0


## 3. Visualize Model Comparison (R² and RMSE)

In [4]:
fig = px.bar(lb, x='Model', y='R2', color='Model', title='Model Comparison: R² Score (Higher is Better)', text_auto='.4f')
fig.update_layout(yaxis=dict(range=[0.55, 0.65]))
fig.show()

fig = px.bar(lb, x='Model', y='RMSE', color='Model', title='Model Comparison: RMSE (Lower is Better)', text_auto='.4f')
fig.update_layout(yaxis=dict(range=[9.5, 11.0]))
fig.show()

## Conclusion
**CatBoost** achieves the highest R² (0.6047) and the lowest RMSE (10.01) and MAE (1.42). It was confidently chosen as the production model over LightGBM and XGBoost.